# Tutorial - Creating a rotor to parameter identification

This tutorial presents the basics of using the ROSS Identification module. In this framework, the process is streamlined through a model-based approach:

Model Conversion: The identification process is performed directly from a provided deterministic rotor. This allows the algorithm to use the existing model structure to identify unknown parameters by matching the numerical response with experimental data.

Below, we will demonstrate how to configure this workflow: Setting up a model for identification.

In [1]:
import ross.identification as irs
import ross as rs
import numpy as np

## Section 1 - Model-Based Identification

The core of this module is performing identification directly from a provided model. In this method, we use a previously configured deterministic rotor as the basis for the identification process. This approach is ideal for those who already have validated models in ROSS and want to identify unknown parameters without rebuilding the system from scratch.

To demonstrate this process, let's start by importing an example rotor:

In [ ]:
rotor = rs.rotor_example()
rotor.plot_rotor()

Defining Parameters for Identification

After providing the model, the next step is to define which variables will be identified. The ROSS Identification module offers two flexible approaches for this configuration:

1. Dictionary Assignment (Tag Mapping)In this method, element tags are used as keys in a dictionary. The values associated with each key are lists containing the specific parameters to be identified for that particular element.Advantage: Total and granular control over which parameters from which elements are included in the process.

2. List Assignment (Automatic Selection)This approach is ideal for identifying the same set of parameters across multiple elements simultaneously (e.g., the stiffness values $k_{xx}$ and $k_{yy}$ of all bearings in the system).Advantage: Automation and speed, eliminating the need to manually list each element and its respective tags when performing a global identification.

In [3]:
rotor_id_1 = irs.make_identification(rotor, {'Bearing 0': ['kxx', 'kyy']}, error = 10/100)
rotor_id_2 = irs.make_identification(rotor, ['kxx', 'kyy'], error = 10/100)

## Section 2 - Running the Analyses

To validate the identification process, we need a dataset that represents the real behavior of the system. For this, we will use the deterministic rotor to simulate experimental FRF (Frequency Response Function) data.

This simulated data will act as our "field measurement," allowing us to test the effectiveness of the Stochastic ROSS in recovering the original parameters from a known response.

In [4]:
# --- Frequency Response Calculation ---
freq_range = np.linspace(500, 1050, 500)  # rad/s
results = rotor.run_freq_response(freq_range)

data = []
inputs_nodes = [3, 20, 23, 26, 29, 32, 35]
outputs_nodes = [5, 15, 40]
probes = []

for i in inputs_nodes:
    for j in outputs_nodes:
        # Extracting FRFs for X and Y directions
        freqsx = results.freq_resp[i*6, j*6]
        freqsy = results.freq_resp[i*6+1, j*6+1]

        data.append(freqsx)
        data.append(freqsy)

        # Mapping probes (input, output, orientation)
        probes.append([i, j, 0])
        probes.append([i, j, 90])

clean_data = np.array(data)

# --- Noise Insertion (SNR Approach) ---
snr_db = 40  # Signal-to-Noise Ratio in dB

# 1. Calculate Signal Power (Mean of squared magnitude)
signal_power = np.mean(np.abs(clean_data)**2)

# 2. Calculate Noise Power based on SNR
# SNR_linear = P_signal / P_noise -> P_noise = P_signal / 10**(SNR_dB / 10)
snr_linear = 10**(snr_db / 10)
noise_power = signal_power / snr_linear

# 3. Generate Complex Gaussian Noise
# We divide noise power by 2 to distribute it between Real and Imaginary parts
noise_std = np.sqrt(noise_power / 2)

noise = (np.random.normal(0, noise_std, clean_data.shape) +
         1j * np.random.normal(0, noise_std, clean_data.shape))

# 4. Final Experimental Data (Clean Signal + Noise)
experimental_data = clean_data + noise

Running the Identification Process
With the experimental data (including noise) and the identification model ready, we proceed to the analysis phase. For this, we use the Identification class (or rotor_id_2), which is responsible for matching the experimental FRFs with the numerical model responses.

The process is structured into two main stages:

Parameter Identification: Finding the best-fit values that minimize the error between the simulated and experimental data. This step determines the most likely values for the parameters selected earlier.

Uncertainty Evaluation: Once the parameters are identified, the system evaluates the reliability of these results. By analyzing how variations in the input parameters affect the model output, we can infer the confidence intervals and the sensitivity of the identification.

In [5]:
# Running the identification process
# Note: We are using 5 iterations and a population of 2 for educational purposes only,
# to ensure the code runs quickly during this tutorial.
# For a real-world application or a precise optimization, these values must be
# significantly increased to ensure convergence and accuracy.
best_fit = rotor_id_1.run_idFreq(
    experimental_data,
    probes,
    freq_range,
    it=5,
    npop=15
)

KeyboardInterrupt: 

In [ ]:
best_fit.plot_magnitude(probe_index = 3, scale_type = 'log')

Uncertainty Inference via SMC-ABC
After identifying the best-fit parameters, we use the Sequential Monte Carlo Approximate Bayesian Computation (SMC-ABC) method to infer the parameter distributions. To run this analysis, the algorithm requires a few specific inputs:

Experimental Data & Probes: The same FRF data and sensor mapping used in the deterministic identification.

Tolerance Thresholds: A list containing the target error levels (e.g., [0.5, 0.25]). The algorithm uses these values to decide which simulated particles are "close enough" to the experimental data to be accepted.

Search Range: A defined interval (e.g., [0.75, 1.25]) that acts as a multiplier for the identified best-fit values, establishing the boundaries where the algorithm will look for uncertainties.

Sample Size (nsample): The number of valid particles to be generated in each step.

By combining these inputs, the method iteratively filters the parameter space to find the distribution that best represents the system's uncertainty.

In [ ]:
# Inferring parameter uncertainties using SMC-ABC
# Note: A low number of samples (nsample=20) and few populations/generations
# were chosen here for demonstration purposes to ensure fast execution.
# For a robust statistical analysis, these values should be significantly higher.
#
# Additionally, to show convergence within this tutorial, we used a wide
# search range and higher intermediate step settings.
uncertainties = rotor_id_1.run_SMC_ABCFreq(
    experimental_data,
    probes,
    freq_range,
    # [min_threshold, max_threshold]
    [0.85, 0.7],
    # [search_range_min, search_range_max]
    [0.75, 1.25],
    nsample=20
)

In [ ]:
uncertainties.plot_magnitude(probe_index = 3,  conf_interval=[95], scale_type = 'log')

In [ ]:
uncertainties.plot_histogram(elements = ['Bearing 0'], parameters = ['kxx'], histogram_kwargs_prior= {'opacity': 0.30, 'marker_color': 'red', 'histnorm':'density'}, histogram_kwargs_posterior= {'histnorm':'density'})